# 04 — Compile Benchmark Results

**Purpose:** Read the per-format, per-size metrics written to `/Volumes/{catalog}/{schema}/{volume}/artifacts/` and compile the full head-to-head comparison across the benchmark.

**Artifacts consumed:**

| Artifact | Source | Content |
|----------|--------|---------|
| `delta_{size}.json` | `01_create_datasets` (`format=delta_pathref`) | Write + ETL metrics (path-ref Delta) |
| `lance_{size}.json` | `01_create_datasets` (`format=lance`) | Write + ETL metrics (Lance) |
| `delta_inline_{size}.json` | `01_create_datasets` (`format=delta_inline`) | Write + ETL metrics (inline Delta anti-pattern) |
| `training_{size}.json` | `03_training_benchmark` | Training throughput: dummy + real, per format |
| `lance_convert_{size}.json` | `optional/01_lance_conversion` (optional) | Migration cost from existing files |

Each write/ETL artifact carries a normalized `common` block with identical keys so they stack into one table with no per-format key mapping. Training metrics are keyed by `formats[fmt][mode]`.

**Sections:**
1. Write & ETL comparison (the decisive Lance advantage — all formats written on the same Spark engine, so the only variable is the storage format)
2. Training throughput comparison (streaming read)
3. Consolidated summary matching the README key-highlights tables

In [0]:
# ── Widgets ─────────────────────────────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "jon_cheung", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")

size    = dbutils.widgets.get("size")
catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")
volume  = dbutils.widgets.get("volume")

base_vol      = f"/Volumes/{catalog}/{schema}/{volume}"
artifacts_dir = f"{base_vol}/artifacts"
print(f"Artifacts dir: {artifacts_dir}")
print(f"Size tier    : {size}")

In [0]:
import json, os
from pathlib import Path

def load_artifact(name):
    """Load a JSON artifact; returns None if it doesn't exist."""
    path = f"{artifacts_dir}/{name}"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

# ── Write/ETL artifacts (from 01_create_datasets, optional conversion) ────────
delta_metrics  = load_artifact(f"delta_{size}.json")
lance_metrics  = load_artifact(f"lance_{size}.json")
inline_metrics = load_artifact(f"delta_inline_{size}.json")
convert_metrics = load_artifact(f"lance_convert_{size}.json")

# ── Training artifact (from 03) ─────────────────────────────────────────
training_metrics = load_artifact(f"training_{size}.json")

loaded = {k: v is not None for k, v in {
    "delta": delta_metrics, "lance": lance_metrics,
    "delta_inline": inline_metrics, "lance_convert": convert_metrics,
    "training": training_metrics,
}.items()}
print(f"Artifacts loaded: {loaded}")

In [0]:
import pandas as pd

# ── Section 1: Write & ETL comparison ───────────────────────────────────────
# Stack the common blocks from all write/ETL artifacts into one comparison table.
# All formats written on the same Spark engine (01_create_datasets) — the only variable
# across these rows is the storage format.
write_artifacts = [
    ("Lance",             lance_metrics),
    ("Delta (path-ref)",  delta_metrics),
    ("Delta (inline)",    inline_metrics),
    ("Lance (conversion)", convert_metrics),
]

def _gb(v):   # on_disk_bytes may be None if physical files can't be located
    return round(v / 1e9, 3) if v is not None else None

rows = []
for label, m in write_artifacts:
    if m is None:
        continue
    c = m["common"]
    rows.append({
        "Format": label,
        "Write time (s)": c["write_total_s"],
        "Target write (s)": c["target_write_s"],
        "Output files": c["n_output_files"],
        "On-disk (GB)": _gb(c["on_disk_bytes"]),
        "ETL backfill (s)": c["etl_backfill_s"],
        "ETL bytes written (MB)": round(c["etl_bytes_written"] / 1e6, 1),
        "Round-trip OK": c["roundtrip_ok"],
    })

if rows:
    write_df = pd.DataFrame(rows)
    print(f"\n═══ WRITE & ETL COMPARISON — size={size} ═══")
    display(write_df)
else:
    print("No write/ETL artifacts found. Run 01_create_datasets first.")

In [0]:
# ── Section 2: Training throughput comparison ───────────────────────────────
if training_metrics:
    fmt_data = training_metrics["formats"]
    train_rows = []
    for fmt, modes in fmt_data.items():
        for mode, metrics in modes.items():
            train_rows.append({
                "Format": fmt,
                "Mode": mode,
                "Samples/sec": metrics.get("samples_per_sec"),
                "Epoch wall (s)": metrics.get("epoch_wall_s"),
                "TTFB (s)": metrics.get("time_to_first_batch_s"),
                "Batch ms p50": metrics.get("batch_ms_p50"),
                "Batch ms p95": metrics.get("batch_ms_p95"),
                "Wait ms p50": metrics.get("wait_ms_p50"),
                "Wait ms p95": metrics.get("wait_ms_p95"),
            })

    if train_rows:
        train_df = pd.DataFrame(train_rows)
        print(f"\n═══ TRAINING THROUGHPUT — size={size}, "
              f"{training_metrics['num_gpu_workers']} GPU workers, "
              f"{training_metrics['num_epochs']} epochs ═══")
        display(train_df)

        # Relative throughput vs Lance (the baseline in the README highlights)
        lance_real = fmt_data.get("lance", {}).get("real", {})
        if lance_real:
            lance_sps = lance_real.get("samples_per_sec", 1)
            print("\n── Relative throughput (real mode, vs Lance baseline) ──")
            for fmt, modes in fmt_data.items():
                real = modes.get("real", {})
                if real and real.get("samples_per_sec"):
                    ratio = real["samples_per_sec"] / lance_sps
                    print(f"  {fmt:20s}: {ratio:.2f}x Lance  ({real['samples_per_sec']:.0f} samples/sec)")
else:
    print("No training artifact found. Run 03_training_benchmark first.")

In [0]:
# ── Section 3: Consolidated summary (README key-highlights format) ───────────
print(f"\n{'='*72}")
print(f"  BENCHMARK SUMMARY — size tier: {size}")
print(f"{'='*72}")

# Write scalability — render whatever write/ETL artifacts are present, rather than
# requiring the Lance + path-ref pair (an inline-only run has no delta_{size}.json).
_write_arms = [
    ("Lance",            lance_metrics),
    ("Delta (path-ref)", delta_metrics),
    ("Delta (inline)",   inline_metrics),
]
_present = [(label, m) for label, m in _write_arms if m is not None]
if _present:
    print(f"\n┌─ WRITE & ETL ───────────────────────")
    for label, m in _present:
        c = m["common"]
        print(f"│  {label:20s}: {c['write_total_s']:>7.1f}s  |  {c['n_output_files']:>6,} output files")
    print(f"│")
    print(f"│  ETL backfill (add column):")
    for label, m in _present:
        c = m["common"]
        note = " (new col only)" if label == "Lance" else (
            " (drags image bytes)" if label == "Delta (inline)" else "")
        print(f"│    {label:18s}: {c['etl_backfill_s']:>7.1f}s  |  {c['etl_bytes_written']/1e6:>8.1f} MB written{note}")
    print(f"└{'─'*60}")
else:
    print("\n(No write/ETL artifacts found — run 01_create_datasets for this size.)")

# Training throughput
if training_metrics:
    fmt_data = training_metrics["formats"]
    print(f"\n┌─ TRAINING THROUGHPUT (streaming shuffle) ──────────────────────────")
    lance_real_sps = fmt_data.get("lance", {}).get("real", {}).get("samples_per_sec")
    for fmt in ["lance", "delta_inline", "delta"]:
        real = fmt_data.get(fmt, {}).get("real", {})
        dummy = fmt_data.get(fmt, {}).get("dummy", {})
        if not real:
            continue
        sps = real.get("samples_per_sec", 0)
        ttfb = real.get("time_to_first_batch_s", 0)
        ratio_str = ""
        if lance_real_sps and fmt != "lance":
            ratio = sps / lance_real_sps
            ratio_str = f"  ({ratio:.2f}x Lance)"
        print(f"│  {fmt:20s}: {sps:>8.0f} samples/sec  | TTFB {ttfb:.2f}s{ratio_str}")
    print(f"└{'─'*60}")

print(f"\nArtifacts dir: {artifacts_dir}")
print("Done.")

## Section 4 — Write & backfill scaling chart (across all tiers)

Unlike the sections above (which use the single `size` widget), this reads **every** tier's
artifacts so the *scaling trend* is visible — the whole point of the chart. Missing artifacts
are skipped, so path-ref (intentionally not run at 1M) simply stops at 100k.

- **Color = format** — blue Lance, Databricks-red Delta (inline), magenta Delta (path-ref)
- **Line style = operation** — solid write, dashed backfill
- **Log y-axis** so all series stay legible across the ~100× range

Saves `artifacts/write_backfill_scaling.png` (light mode, for the README).

In [0]:
# ── Section 4: Write & backfill scaling chart (reads ALL tiers) ──────────────
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

TIERS       = ["10k", "100k", "1m"]        # x-axis, in scale order
TIER_LABELS = ["10k", "100k", "1M"]

# Validated CVD-safe palette (light mode); Databricks red for inline.
FORMATS = [                                # (short, legend label, artifact prefix, color)
    ("Lance",    "Lance",            "lance",        "#1E6FB4"),
    ("inline",   "Delta (inline)",   "delta_inline", "#FF3621"),
    ("path-ref", "Delta (path-ref)", "delta",        "#B84A9E"),
]
OPS = [                                     # (metric key, line style, op label)
    ("write_total_s",  "-",  "write"),
    ("etl_backfill_s", "--", "backfill"),
]

def _series(prefix, metric_key):
    """(xs, ys) for one format/metric across tiers; skips missing artifacts."""
    xs, ys = [], []
    for i, t in enumerate(TIERS):
        m = load_artifact(f"{prefix}_{t}.json")
        if m is not None:
            xs.append(i); ys.append(m["common"][metric_key])
    return xs, ys

def _bytes_series(prefix, key="etl_bytes_written"):
    """(xs, ys) for a bytes metric across tiers, in MB."""
    xs, ys = [], []
    for i, t in enumerate(TIERS):
        m = load_artifact(f"{prefix}_{t}.json")
        if m is not None:
            xs.append(i); ys.append(m["common"][key] / 1e6)  # → MB
    return xs, ys

# ── Two-panel figure: wall-clock time (left) + bytes written (right) ──────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), dpi=150)
fig.subplots_adjust(top=0.74, wspace=0.35)

# Panel 1: Write & backfill wall-clock time
for short, _lbl, prefix, color in FORMATS:
    for metric_key, style, op in OPS:
        xs, ys = _series(prefix, metric_key)
        if not xs:
            continue
        ax1.plot(xs, ys, style, color=color, linewidth=2, marker="o", markersize=5.5,
                markeredgecolor="white", markeredgewidth=1.2, zorder=3)
        ax1.annotate(f" {short} {op}", xy=(xs[-1], ys[-1]), va="center",
                    fontsize=8.5, color=color)

ax1.set_yscale("log")
ax1.set_yticks([2, 5, 10, 20, 50, 100, 200])
ax1.set_yticklabels(["2s", "5s", "10s", "20s", "50s", "100s", "200s"])
ax1.set_xticks(range(len(TIERS)))
ax1.set_xticklabels(TIER_LABELS)
ax1.set_xlim(-0.15, len(TIERS) - 1 + 0.95)
ax1.set_xlabel("Dataset size (rows)")
ax1.set_ylabel("Seconds (↓ better)")
ax1.set_title("Wall-clock time", fontsize=11, fontweight="bold", loc="left")
ax1.grid(axis="y", color="#e6e4df", linewidth=1)
for spine in ("top", "right", "left"):
    ax1.spines[spine].set_visible(False)
ax1.tick_params(length=0)

# Panel 2: Bytes written — solid = initial write, dashed = backfill
for short, _lbl, prefix, color in FORMATS:
    # Collect both series to detect overlap
    xs_w, ys_w = _bytes_series(prefix, "on_disk_bytes")
    xs_b, ys_b = _bytes_series(prefix, "etl_bytes_written")

    # Solid: initial write (on_disk_bytes)
    xs_w, ys_w = _bytes_series(prefix, "on_disk_bytes")
    if xs_w:
        ax2.plot(xs_w, ys_w, "-", color=color, linewidth=2, marker="o", markersize=5.5,
                 markeredgecolor="white", markeredgewidth=1.2, zorder=3)
        ax2.annotate(f"{short} write", xy=(xs_w[-1], ys_w[-1]),
                     xytext=(5, 0), textcoords="offset points",
                     va="center", fontsize=8.5, color=color)
    # Dashed: backfill (etl_bytes_written)
    xs_b, ys_b = _bytes_series(prefix, "etl_bytes_written")
    if xs_b:
        ax2.plot(xs_b, ys_b, "--", color=color, linewidth=2, marker="o", markersize=5.5,
                 markeredgecolor="white", markeredgewidth=1.2, zorder=3)
        # Offset inline backfill down to avoid overlap with inline write
        y_offset = -6 if (xs_w and abs(ys_w[-1] - ys_b[-1]) / max(ys_w[-1], ys_b[-1]) < 0.5) else 0
        ax2.annotate(f"{short} backfill", xy=(xs_b[-1], ys_b[-1]),
                     xytext=(5, y_offset), textcoords="offset points",
                     va="center", fontsize=8.5, color=color)

ax2.set_yscale("log")
ax2.set_yticks([0.1, 1, 10, 100, 1000, 10000, 100000])
ax2.set_yticklabels(["0.1 MB", "1 MB", "10 MB", "100 MB", "1 GB", "10 GB", "100 GB"])
ax2.set_xticks(range(len(TIERS)))
ax2.set_xticklabels(TIER_LABELS)
ax2.set_xlim(-0.15, len(TIERS) - 1 + 0.95)
ax2.set_xlabel("Dataset size (rows)")
ax2.set_ylabel("Data volume (↓ better)")
ax2.set_title("Bytes written", fontsize=11, fontweight="bold", loc="left")
ax2.grid(axis="y", color="#e6e4df", linewidth=1)
for spine in ("top", "right", "left"):
    ax2.spines[spine].set_visible(False)
ax2.tick_params(length=0)

# Suptitle + shared legend
fig.suptitle("Write & backfill scaling — Lance vs Delta", fontsize=13, fontweight="bold",
             x=0.09, ha="left", y=0.98)

color_handles = [Line2D([0], [0], color=c, lw=2.5, label=lbl) for _s, lbl, _p, c in FORMATS]
style_handles = [Line2D([0], [0], color="#8a8780", lw=2, ls="-",  label="Write (solid)"),
                 Line2D([0], [0], color="#8a8780", lw=2, ls="--", label="Backfill (dashed)")]
ax1.legend(handles=color_handles + style_handles, loc="lower left", frameon=False,
           fontsize=9, ncol=5, bbox_to_anchor=(0, 1.10), columnspacing=1.5)

fig.text(0.09, 0.01,
         "Wall-clock seconds (log scale). All formats written on one 8-worker Spark cluster. "
         "Path-ref not run at 1M. Bytes = backfill only (new derived column).",
         fontsize=8.5, color="#8a8780")

out_png = f"{artifacts_dir}/write_backfill_scaling.png"
# fig.savefig(out_png, bbox_inches="tight", facecolor="white")
# print(f"Saved {out_png}")
display(fig)
plt.close(fig)

# ── Summary table: Write & Backfill (all tiers) ──────────────────────────────
write_rows = []
for tier in TIERS:
    for label, prefix in [("Lance", "lance"), ("Delta (inline)", "delta_inline"), ("Delta (path-ref)", "delta")]:
        m = load_artifact(f"{prefix}_{tier}.json")
        if m is None:
            write_rows.append({"Format": label, "Size": tier,
                "Write (s)": "~", "Backfill (s)": "~", "Write (MB)": "~", "Backfill (MB)": "~"})
        else:
            c = m["common"]
            write_rows.append({"Format": label, "Size": tier,
                "Write (s)": f"{c['write_total_s']:.1f}",
                "Backfill (s)": f"{c['etl_backfill_s']:.1f}",
                "Write (MB)": f"{c['on_disk_bytes'] / 1e6:,.0f}",
                "Backfill (MB)": f"{c['etl_bytes_written'] / 1e6:,.1f}"})

print("\n═══ WRITE & BACKFILL (all tiers) ═══")
display(pd.DataFrame(write_rows))

## Section 5 — Training throughput scaling chart (across all tiers)

Reads **every** tier's training artifact to show the scaling trend for the
streaming Ray Data → Ray Train read path. Same style as the write/backfill chart.

- **Color = format** — blue Lance, Databricks-red Delta (inline), magenta Delta (path-ref)
- **Line style = metric** — solid samples/sec, dashed TTFB
- **Log y-axis** for both panels

Key README finding: inline Delta ties Lance on throughput; path-ref is the read-side loser
with widening TTFB gap at scale.

In [0]:
# ── Section 5: Training throughput scaling chart (reads ALL tiers) ────────────
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker

TRAIN_TIERS       = ["10k", "100k", "1m"]
TRAIN_TIER_LABELS = ["10k", "100k", "1M"]

# Same palette as write/backfill chart
TRAIN_FORMATS = [
    ("Lance",    "Lance",            "lance",        "#1E6FB4"),
    ("inline",   "Delta (inline)",   "delta_inline", "#FF3621"),
    ("path-ref", "Delta (path-ref)", "delta",        "#B84A9E"),
]

def _train_series(fmt_key, metric_key):
    """(xs, ys) for one format/metric across tiers from training artifacts."""
    xs, ys = [], []
    for i, t in enumerate(TRAIN_TIERS):
        m = load_artifact(f"training_{t}.json")
        if m is None:
            continue
        real = m.get("formats", {}).get(fmt_key, {}).get("real", {})
        val = real.get(metric_key)
        if val is not None:
            xs.append(i); ys.append(val)
    return xs, ys

# ── Two-panel figure: throughput (left) + TTFB (right) ────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), dpi=150)
fig.subplots_adjust(top=0.78, wspace=0.35)

# Panel 1: Samples/sec (higher is better)
for short, _lbl, fmt_key, color in TRAIN_FORMATS:
    xs, ys = _train_series(fmt_key, "samples_per_sec")
    if not xs:
        continue
    ax1.plot(xs, ys, "-", color=color, linewidth=2, marker="o", markersize=5.5,
             markeredgecolor="white", markeredgewidth=1.2, zorder=3)
    ax1.annotate(f" {short}", xy=(xs[-1], ys[-1]), va="center",
                 fontsize=8.5, color=color)

ax1.set_yscale("log")
ax1.set_ylim(top=300)
ax1.set_yticks([50, 100, 150, 200, 300])
ax1.get_yaxis().set_major_formatter(mticker.ScalarFormatter())
ax1.set_xticks(range(len(TRAIN_TIERS)))
ax1.set_xticklabels(TRAIN_TIER_LABELS)
ax1.set_xlim(-0.15, len(TRAIN_TIERS) - 1 + 0.7)
ax1.set_xlabel("Dataset size (rows)")
ax1.set_ylabel("Samples / sec (↑ better)")
ax1.set_title("Training throughput", fontsize=11, fontweight="bold", loc="left")
ax1.grid(axis="y", color="#e6e4df", linewidth=1)
for spine in ("top", "right", "left"):
    ax1.spines[spine].set_visible(False)
ax1.tick_params(length=0)

# Panel 2: TTFB (lower is better)
for short, _lbl, fmt_key, color in TRAIN_FORMATS:
    xs, ys = _train_series(fmt_key, "time_to_first_batch_s")
    if not xs:
        continue
    ax2.plot(xs, ys, "-", color=color, linewidth=2, marker="o", markersize=5.5,
             markeredgecolor="white", markeredgewidth=1.2, zorder=3)
    ax2.annotate(f" {short}", xy=(xs[-1], ys[-1]), va="center",
                 fontsize=8.5, color=color)

ax2.set_yscale("log")
ax2.set_yticks([5, 10, 20, 50, 100, 200, 500])
ax2.set_yticklabels(["5s", "10s", "20s", "50s", "100s", "200s", "500s"])
ax2.set_xticks(range(len(TRAIN_TIERS)))
ax2.set_xticklabels(TRAIN_TIER_LABELS)
ax2.set_xlim(-0.15, len(TRAIN_TIERS) - 1 + 0.7)
ax2.set_xlabel("Dataset size (rows)")
ax2.set_ylabel("Time to first batch (↓ better)")
ax2.set_title("TTFB (idle GPU time)", fontsize=11, fontweight="bold", loc="left")
ax2.grid(axis="y", color="#e6e4df", linewidth=1)
for spine in ("top", "right", "left"):
    ax2.spines[spine].set_visible(False)
ax2.tick_params(length=0)

# Suptitle + shared legend
fig.suptitle("Training read scaling — Lance vs Delta (streaming Ray Data)",
             fontsize=13, fontweight="bold", x=0.09, ha="left", y=0.98)

color_handles = [Line2D([0], [0], color=c, lw=2.5, label=lbl)
                 for _s, lbl, _p, c in TRAIN_FORMATS]
leg1 = ax1.legend(handles=color_handles, loc="lower left", frameon=False,
                  fontsize=9, ncol=3, bbox_to_anchor=(0, 1.05))

fig.text(0.09, 0.01,
         "Streaming Ray Data → Ray Train DDP, real mode (image decode + forward pass). "
         "Path-ref not run at 1M.",
         fontsize=8.5, color="#8a8780")

out_png = f"{artifacts_dir}/training_scaling.png"
# fig.savefig(out_png, bbox_inches="tight", facecolor="white")
# print(f"Saved {out_png}")
display(fig)
plt.close(fig)

# ── Summary table: Training Read (all tiers) ─────────────────────────────────
train_rows = []
for tier in TRAIN_TIERS:
    for label, fmt_key in [("Lance", "lance"), ("Delta (inline)", "delta_inline"), ("Delta (path-ref)", "delta")]:
        m = load_artifact(f"training_{tier}.json")
        real = m.get("formats", {}).get(fmt_key, {}).get("real", {}) if m else {}
        if not real:
            train_rows.append({"Format": label, "Size": tier, "Samples/s": "~", "TTFB (s)": "~"})
        else:
            train_rows.append({"Format": label, "Size": tier,
                "Samples/s": str(round(real["samples_per_sec"])),
                "TTFB (s)": f"{real['time_to_first_batch_s']:.1f}"})

print("\n═══ TRAINING READ — streaming throughput (all tiers) ═══")
display(pd.DataFrame(train_rows))